In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
NVIDIA H100 80GB HBM3


In [7]:
import torch
from transformer_lens.model_bridge import TransformerBridge


# Loading GPT-2
device = "cuda" if torch.cuda.is_available() else "cpu"

model = TransformerBridge.boot_transformers("openai-community/gpt2", device=device)
model.enable_compatibility_mode(disable_warnings=True)  # for legacy HookedTransformer-equivalent numerics

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 3717.26it/s]


In [8]:
# Generating predictions
prompt = "The capital of France is"
tokens = model.to_tokens(prompt)

print(tokens)
print(tokens.shape)
print(model.to_str_tokens(tokens))
logits = model(tokens)
print(logits.shape)

tensor([[50256,   464,  3139,   286,  4881,   318]], device='cuda:0')
torch.Size([1, 6])
['<|endoftext|>', 'The', ' capital', ' of', ' France', ' is']
torch.Size([1, 6, 50257])


In [10]:
# Understand tokens/logits
text = "The capital of France is"

logits = model(text)

print("\nLogits shape:", logits.shape)

probs = logits[0, -1].softmax(dim=-1)
top_probs, top_tokens = probs.topk(10)

print("\nTop predictions:")
for prob, token in zip(top_probs, top_tokens):
    print(f"{model.to_string(token)!r}: {prob.item():.4f}")


Logits shape: torch.Size([1, 6, 50257])

Top predictions:
' now': 0.0475
' the': 0.0374
' a': 0.0355
' home': 0.0309
' in': 0.0270
' under': 0.0257
' being': 0.0209
' set': 0.0180
' on': 0.0168
' not': 0.0149


In [11]:
gpt2_text = "Natural language processing tasks, such as question answering, machine translation, reading comprehension, and summarization, are typically approached with supervised learning on taskspecific datasets."
gpt2_tokens = model.to_tokens(gpt2_text)
print(gpt2_tokens.device)
gpt2_logits, gpt2_cache = model.run_with_cache(gpt2_tokens, remove_batch_dim=True)

cuda:0


In [12]:
print(type(gpt2_cache))
attention_pattern = gpt2_cache["pattern", 0, "attn"]
print(attention_pattern.shape)
gpt2_str_tokens = model.to_str_tokens(gpt2_text)

<class 'transformer_lens.ActivationCache.ActivationCache'>
torch.Size([12, 33, 33])


In [14]:
import circuitsvis as cv


print("Layer 0 Head Attention Patterns:")
cv.attention.attention_heads(tokens=gpt2_str_tokens, attention=attention_pattern)

Layer 0 Head Attention Patterns:
